In [0]:
from pyspark.sql.functions import col, when, to_date, expr

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

results = spark.table("workspace.bronze.results_raw")

matches_clean = (
    results
    .withColumn("date", to_date(col("date")))
    .withColumn("home_score", expr("try_cast(home_score as int)"))
    .withColumn("away_score", expr("try_cast(away_score as int)"))
    .withColumn("neutral", col("neutral").cast("boolean"))
    .dropDuplicates()
)

historical_matches = (
    matches_clean
    .filter(col("home_score").isNotNull() & col("away_score").isNotNull())
    .withColumn(
        "result",
        when(col("home_score") > col("away_score"), "home_win")
        .when(col("home_score") < col("away_score"), "away_win")
        .otherwise("draw")
    )
    .withColumn("goal_diff", col("home_score") - col("away_score"))
)

worldcup_2026_group_matches = (
    matches_clean
    .filter(
        (col("tournament") == "FIFA World Cup") &
        (col("home_score").isNull()) &
        (col("away_score").isNull())
    )
)

historical_matches.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.matches_clean")
worldcup_2026_group_matches.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.worldcup_2026_group_matches")

print("Tablas Silver creadas correctamente.")

In [0]:
display(spark.table("workspace.silver.matches_clean").limit(10))

In [0]:
display(spark.table("workspace.silver.worldcup_2026_group_matches").limit(20))

In [0]:
spark.sql("SHOW TABLES IN workspace.silver").show()

In [0]:
print("Partidos históricos:", spark.table("workspace.silver.matches_clean").count())
print("Partidos Mundial 2026:", spark.table("workspace.silver.worldcup_2026_group_matches").count())